# **Library Management System**

In [1]:
"""
Library Management System
==========================

A professional, menu-driven Library Management System built with core
Object-Oriented Programming principles (encapsulation, inheritance,
composition, abstraction and polymorphism).

The module is self-contained and relies only on the Python standard
library, so it can be run as-is in Google Colab, a local terminal, or
any standard Python 3.9+ interpreter.

Author: (Your Name)
License: MIT
"""

from __future__ import annotations

import uuid
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Dict, List, Optional


# ---------------------------------------------------------------------------
# Custom Exception Hierarchy
# ---------------------------------------------------------------------------
class LibraryError(Exception):
    """Base class for all library-related exceptions."""


class InvalidInputError(LibraryError):
    """Raised when user-supplied data fails validation."""


class BookNotFoundError(LibraryError):
    """Raised when a requested book (by ISBN) does not exist."""


class DuplicateBookError(LibraryError):
    """Raised when attempting to register a book with a duplicate ISBN."""


class BookNotAvailableError(LibraryError):
    """Raised when there are no available copies of a requested book."""


class MemberNotFoundError(LibraryError):
    """Raised when a requested member (by member ID) does not exist."""


class DuplicateMemberError(LibraryError):
    """Raised when attempting to register a member with a duplicate email."""


class BorrowLimitExceededError(LibraryError):
    """Raised when a member attempts to exceed their borrowing limit."""


class LoanNotFoundError(LibraryError):
    """Raised when a requested loan record does not exist."""


class BookAlreadyReturnedError(LibraryError):
    """Raised when attempting to return a book that is already returned."""


# ---------------------------------------------------------------------------
# Enumerations
# ---------------------------------------------------------------------------
class BookGenre(Enum):
    """Supported book genres."""

    FICTION = "Fiction"
    NON_FICTION = "Non-Fiction"
    SCIENCE = "Science"
    TECHNOLOGY = "Technology"
    HISTORY = "History"
    BIOGRAPHY = "Biography"
    OTHER = "Other"


class MembershipType(Enum):
    """
    Membership tiers.

    Each member holds a tuple of (display label, max concurrent books,
    loan period in days) so borrowing rules stay data-driven rather than
    scattered across the codebase (open/closed principle).
    """

    STANDARD = ("Standard", 3, 14)
    PREMIUM = ("Premium", 6, 21)
    STUDENT = ("Student", 4, 14)

    def __init__(self, label: str, max_books: int, loan_days: int) -> None:
        self.label = label
        self.max_books = max_books
        self.loan_days = loan_days


# ---------------------------------------------------------------------------
# Abstraction + Inheritance: Person hierarchy
# ---------------------------------------------------------------------------
class Person(ABC):
    """
    Abstract base class representing any individual associated with the
    library (a member or a librarian).

    Demonstrates encapsulation via private attributes exposed through
    validated properties, and abstraction via the ``get_role`` contract
    that every subclass must fulfil.
    """

    def __init__(self, name: str, email: str, phone: str) -> None:
        self._id: str = str(uuid.uuid4())[:8].upper()
        self.name = name
        self.email = email
        self.phone = phone
        self._created_at: datetime = datetime.now()

    # -- Identity -----------------------------------------------------
    @property
    def id(self) -> str:
        return self._id

    @property
    def created_at(self) -> datetime:
        return self._created_at

    # -- Validated attributes ------------------------------------------
    @property
    def name(self) -> str:
        return self._name

    @name.setter
    def name(self, value: str) -> None:
        if not isinstance(value, str) or not value.strip():
            raise InvalidInputError("Name cannot be empty.")
        self._name = value.strip().title()

    @property
    def email(self) -> str:
        return self._email

    @email.setter
    def email(self, value: str) -> None:
        value = (value or "").strip().lower()
        if "@" not in value or "." not in value.split("@")[-1]:
            raise InvalidInputError(f"Invalid email address: '{value}'")
        self._email = value

    @property
    def phone(self) -> str:
        return self._phone

    @phone.setter
    def phone(self, value: str) -> None:
        digits = (value or "").replace("-", "").replace(" ", "")
        if not digits.isdigit() or not (7 <= len(digits) <= 15):
            raise InvalidInputError(f"Invalid phone number: '{value}'")
        self._phone = value.strip()

    @abstractmethod
    def get_role(self) -> str:
        """Return a human-readable role label. Must be implemented by subclasses."""
        raise NotImplementedError

    def __str__(self) -> str:
        return f"{self.get_role()} | {self.name} (ID: {self.id}, Email: {self.email})"

    def __repr__(self) -> str:
        return f"<{self.__class__.__name__} id={self.id!r} name={self.name!r}>"


class Member(Person):
    """A library member who can borrow and return books."""

    def __init__(
        self,
        name: str,
        email: str,
        phone: str,
        membership_type: MembershipType = MembershipType.STANDARD,
    ) -> None:
        super().__init__(name, email, phone)
        self.membership_type = membership_type
        self._active_loans: Dict[str, "Loan"] = {}  # loan_id -> Loan
        self._history: List["Loan"] = []
        self._outstanding_fine: float = 0.0

    def get_role(self) -> str:
        return "Member"

    @property
    def active_loans(self) -> List["Loan"]:
        return list(self._active_loans.values())

    @property
    def borrowed_count(self) -> int:
        return len(self._active_loans)

    @property
    def outstanding_fine(self) -> float:
        return round(self._outstanding_fine, 2)

    @property
    def history(self) -> List["Loan"]:
        return list(self._history)

    def can_borrow(self) -> bool:
        """A member may borrow if under their tier's limit and has no unpaid fines."""
        return (
            self.borrowed_count < self.membership_type.max_books
            and self._outstanding_fine <= 0.0
        )

    def _attach_loan(self, loan: "Loan") -> None:
        self._active_loans[loan.loan_id] = loan

    def _detach_loan(self, loan: "Loan") -> None:
        self._active_loans.pop(loan.loan_id, None)
        self._history.append(loan)

    def _add_fine(self, amount: float) -> None:
        self._outstanding_fine = round(self._outstanding_fine + amount, 2)

    def pay_fine(self, amount: float) -> float:
        """Pay down the member's outstanding fine. Returns the remaining balance."""
        if amount <= 0:
            raise InvalidInputError("Payment amount must be positive.")
        self._outstanding_fine = round(max(0.0, self._outstanding_fine - amount), 2)
        return self._outstanding_fine


class Librarian(Person):
    """A staff member who administers the library catalogue and loans."""

    def __init__(
        self, name: str, email: str, phone: str, employee_code: Optional[str] = None
    ) -> None:
        super().__init__(name, email, phone)
        self.employee_code = employee_code or f"EMP-{self.id}"

    def get_role(self) -> str:
        return "Librarian"


# ---------------------------------------------------------------------------
# Book (Encapsulation of inventory state)
# ---------------------------------------------------------------------------
class Book:
    """Represents a catalogue title and its copy inventory."""

    def __init__(
        self,
        title: str,
        author: str,
        isbn: str,
        genre: BookGenre = BookGenre.OTHER,
        total_copies: int = 1,
        year: Optional[int] = None,
    ) -> None:
        if not title.strip():
            raise InvalidInputError("Book title cannot be empty.")
        if not author.strip():
            raise InvalidInputError("Author name cannot be empty.")
        if not isbn.strip():
            raise InvalidInputError("ISBN cannot be empty.")
        if total_copies < 1:
            raise InvalidInputError("A book must have at least one copy.")
        if year is not None and (year < 1450 or year > datetime.now().year):
            raise InvalidInputError(f"Invalid publication year: {year}")

        self.title = title.strip()
        self.author = author.strip()
        self.isbn = isbn.strip()
        self.genre = genre
        self.year = year
        self._total_copies = total_copies
        self._available_copies = total_copies

    @property
    def total_copies(self) -> int:
        return self._total_copies

    @property
    def available_copies(self) -> int:
        return self._available_copies

    @property
    def is_available(self) -> bool:
        return self._available_copies > 0

    def add_copies(self, count: int) -> None:
        if count <= 0:
            raise InvalidInputError("Number of copies to add must be positive.")
        self._total_copies += count
        self._available_copies += count

    def _reserve_copy(self) -> None:
        """Internal: decrement available copies when a book is issued."""
        if self._available_copies <= 0:
            raise BookNotAvailableError(f"No available copies of '{self.title}'.")
        self._available_copies -= 1

    def _release_copy(self) -> None:
        """Internal: increment available copies when a book is returned."""
        if self._available_copies >= self._total_copies:
            return  # defensive guard against double-release
        self._available_copies += 1

    def __str__(self) -> str:
        return (
            f"'{self.title}' by {self.author} "
            f"[{self.genre.value}] ISBN:{self.isbn} "
            f"({self.available_copies}/{self.total_copies} available)"
        )


# ---------------------------------------------------------------------------
# Loan (Transaction record)
# ---------------------------------------------------------------------------
@dataclass
class Loan:
    """An immutable-by-convention record of a single borrow transaction."""

    loan_id: str
    isbn: str
    member_id: str
    issue_date: datetime
    due_date: datetime
    return_date: Optional[datetime] = None

    @property
    def is_returned(self) -> bool:
        return self.return_date is not None

    @property
    def is_overdue(self) -> bool:
        reference = self.return_date or datetime.now()
        return reference > self.due_date

    @property
    def days_overdue(self) -> int:
        if not self.is_overdue:
            return 0
        reference = self.return_date or datetime.now()
        return (reference - self.due_date).days

    def __str__(self) -> str:
        status = "RETURNED" if self.is_returned else "ACTIVE"
        overdue_tag = " (OVERDUE)" if not self.is_returned and self.is_overdue else ""
        return (
            f"Loan#{self.loan_id} | ISBN:{self.isbn} | Member:{self.member_id} | "
            f"Due:{self.due_date:%Y-%m-%d} | {status}{overdue_tag}"
        )


# ---------------------------------------------------------------------------
# Library (Composition root / Facade over the whole domain)
# ---------------------------------------------------------------------------
class Library:
    """
    Central coordinator that *composes* the catalogue, membership base and
    loan ledger. All cross-entity business rules (issuing, returning,
    fines) live here so that Book/Member/Loan stay focused on their own
    state (single-responsibility principle).
    """

    FINE_PER_DAY: float = 5.0

    def __init__(self, name: str) -> None:
        self.name = name
        self._books: Dict[str, Book] = {}
        self._members: Dict[str, Member] = {}
        self._librarians: Dict[str, Librarian] = {}
        self._loans: Dict[str, Loan] = {}

    # -- Catalogue management ------------------------------------------
    def add_book(
        self,
        title: str,
        author: str,
        isbn: str,
        genre: BookGenre,
        total_copies: int = 1,
        year: Optional[int] = None,
    ) -> Book:
        if isbn.strip() in self._books:
            existing = self._books[isbn.strip()]
            existing.add_copies(total_copies)
            return existing
        book = Book(title, author, isbn, genre, total_copies, year)
        self._books[book.isbn] = book
        return book

    def get_book(self, isbn: str) -> Book:
        book = self._books.get(isbn.strip())
        if book is None:
            raise BookNotFoundError(f"No book found with ISBN '{isbn}'.")
        return book

    def remove_book(self, isbn: str) -> None:
        book = self.get_book(isbn)
        if book.available_copies != book.total_copies:
            raise LibraryError(
                f"Cannot remove '{book.title}': copies are currently on loan."
            )
        del self._books[book.isbn]

    def search_books(self, keyword: str) -> List[Book]:
        keyword = keyword.strip().lower()
        return [
            b
            for b in self._books.values()
            if keyword in b.title.lower()
            or keyword in b.author.lower()
            or keyword in b.isbn.lower()
        ]

    @property
    def catalogue(self) -> List[Book]:
        return sorted(self._books.values(), key=lambda b: b.title)

    # -- Membership management ------------------------------------------
    def register_member(
        self,
        name: str,
        email: str,
        phone: str,
        membership_type: MembershipType = MembershipType.STANDARD,
    ) -> Member:
        normalized_email = email.strip().lower()
        if any(m.email == normalized_email for m in self._members.values()):
            raise DuplicateMemberError(
                f"A member with email '{normalized_email}' already exists."
            )
        member = Member(name, email, phone, membership_type)
        self._members[member.id] = member
        return member

    def get_member(self, member_id: str) -> Member:
        member = self._members.get(member_id.strip().upper())
        if member is None:
            raise MemberNotFoundError(f"No member found with ID '{member_id}'.")
        return member

    def register_librarian(self, name: str, email: str, phone: str) -> Librarian:
        librarian = Librarian(name, email, phone)
        self._librarians[librarian.id] = librarian
        return librarian

    @property
    def members(self) -> List[Member]:
        return sorted(self._members.values(), key=lambda m: m.name)

    # -- Circulation (issue / return) ------------------------------------
    def issue_book(self, isbn: str, member_id: str) -> Loan:
        book = self.get_book(isbn)
        member = self.get_member(member_id)

        if not book.is_available:
            raise BookNotAvailableError(f"'{book.title}' currently has no available copies.")
        if not member.can_borrow():
            if member.outstanding_fine > 0:
                raise BorrowLimitExceededError(
                    f"{member.name} has an outstanding fine of "
                    f"${member.outstanding_fine:.2f} and cannot borrow until it is paid."
                )
            raise BorrowLimitExceededError(
                f"{member.name} has reached their borrowing limit of "
                f"{member.membership_type.max_books} books."
            )

        book._reserve_copy()
        issue_date = datetime.now()
        due_date = issue_date + timedelta(days=member.membership_type.loan_days)
        loan = Loan(
            loan_id=str(uuid.uuid4())[:8].upper(),
            isbn=book.isbn,
            member_id=member.id,
            issue_date=issue_date,
            due_date=due_date,
        )
        self._loans[loan.loan_id] = loan
        member._attach_loan(loan)
        return loan

    def return_book(self, loan_id: str) -> float:
        """Return a book and return the fine (if any) incurred for the loan."""
        loan = self._loans.get(loan_id.strip().upper())
        if loan is None:
            raise LoanNotFoundError(f"No loan found with ID '{loan_id}'.")
        if loan.is_returned:
            raise BookAlreadyReturnedError(f"Loan '{loan_id}' was already returned.")

        loan.return_date = datetime.now()
        book = self.get_book(loan.isbn)
        member = self.get_member(loan.member_id)

        book._release_copy()
        member._detach_loan(loan)

        fine = 0.0
        if loan.is_overdue:
            fine = round(loan.days_overdue * self.FINE_PER_DAY, 2)
            member._add_fine(fine)
        return fine

    def overdue_loans(self) -> List[Loan]:
        return [
            loan
            for loan in self._loans.values()
            if not loan.is_returned and loan.is_overdue
        ]

    def all_loans(self) -> List[Loan]:
        return sorted(self._loans.values(), key=lambda l: l.issue_date, reverse=True)

    # -- Reporting --------------------------------------------------------
    def statistics(self) -> Dict[str, float]:
        total_books = sum(b.total_copies for b in self._books.values())
        available = sum(b.available_copies for b in self._books.values())
        return {
            "titles": len(self._books),
            "total_copies": total_books,
            "available_copies": available,
            "on_loan": total_books - available,
            "members": len(self._members),
            "active_loans": len([l for l in self._loans.values() if not l.is_returned]),
            "overdue_loans": len(self.overdue_loans()),
        }


# ---------------------------------------------------------------------------
# Command Line Interface
# ---------------------------------------------------------------------------
class LibraryCLI:
    """Menu-driven console front-end for the :class:`Library` domain model."""

    def __init__(self) -> None:
        self.library = Library("City Central Library")
        self._seed_demo_data()

    # -- Bootstrapping ------------------------------------------------
    def _seed_demo_data(self) -> None:
        """Pre-populate the library with a few books/members for convenience."""
        self.library.add_book(
            "Clean Code", "Robert C. Martin", "9780132350884",
            BookGenre.TECHNOLOGY, total_copies=3, year=2008,
        )
        self.library.add_book(
            "Dune", "Frank Herbert", "9780441172719",
            BookGenre.FICTION, total_copies=2, year=1965,
        )
        self.library.add_book(
            "A Brief History of Time", "Stephen Hawking", "9780553380163",
            BookGenre.SCIENCE, total_copies=2, year=1988,
        )
        self.library.register_librarian("Alice Turner", "alice.turner@library.org", "1234567890")

    # -- Main loop ------------------------------------------------------
    def run(self) -> None:
        print(f"\nWelcome to {self.library.name} Management System")
        actions = {
            "1": self._add_book,
            "2": self._search_books,
            "3": self._list_catalogue,
            "4": self._register_member,
            "5": self._issue_book,
            "6": self._return_book,
            "7": self._view_member,
            "8": self._pay_fine,
            "9": self._overdue_report,
            "10": self._statistics,
        }
        while True:
            self._print_menu()
            choice = input("Enter your choice: ").strip()
            if choice == "0":
                print("Goodbye!")
                break
            action = actions.get(choice)
            if action is None:
                print("Invalid choice. Please select a valid menu option.\n")
                continue
            try:
                action()
            except LibraryError as exc:
                print(f"\n[Error] {exc}\n")
            except ValueError:
                print("\n[Error] Please enter a valid numeric value.\n")

    @staticmethod
    def _print_menu() -> None:
        print(
            "\n"
            "==================== MAIN MENU ====================\n"
            " 1. Add / Restock Book\n"
            " 2. Search Books\n"
            " 3. View Full Catalogue\n"
            " 4. Register Member\n"
            " 5. Issue Book\n"
            " 6. Return Book\n"
            " 7. View Member Details\n"
            " 8. Pay Outstanding Fine\n"
            " 9. Overdue Loans Report\n"
            "10. Library Statistics\n"
            " 0. Exit\n"
            "===================================================="
        )

    # -- Menu actions -----------------------------------------------------
    def _add_book(self) -> None:
        print("\n-- Add / Restock Book --")
        title = input("Title: ").strip()
        author = input("Author: ").strip()
        isbn = input("ISBN: ").strip()
        print("Genres:", ", ".join(g.value for g in BookGenre))
        genre_input = input("Genre: ").strip().title()
        genre = next((g for g in BookGenre if g.value.lower() == genre_input.lower()), BookGenre.OTHER)
        copies = int(input("Number of copies: ").strip() or "1")
        year_raw = input("Publication year (optional): ").strip()
        year = int(year_raw) if year_raw else None

        book = self.library.add_book(title, author, isbn, genre, copies, year)
        print(f"\nSuccess: {book}")

    def _search_books(self) -> None:
        keyword = input("\nSearch by title, author, or ISBN: ").strip()
        results = self.library.search_books(keyword)
        if not results:
            print("No matching books found.")
            return
        print(f"\nFound {len(results)} result(s):")
        for book in results:
            print(f"  - {book}")

    def _list_catalogue(self) -> None:
        catalogue = self.library.catalogue
        if not catalogue:
            print("\nThe catalogue is currently empty.")
            return
        print(f"\n-- Full Catalogue ({len(catalogue)} titles) --")
        for book in catalogue:
            print(f"  - {book}")

    def _register_member(self) -> None:
        print("\n-- Register New Member --")
        name = input("Full name: ").strip()
        email = input("Email: ").strip()
        phone = input("Phone: ").strip()
        print("Membership types:", ", ".join(m.label for m in MembershipType))
        type_input = input("Membership type [Standard]: ").strip().title() or "Standard"
        membership = next(
            (m for m in MembershipType if m.label.lower() == type_input.lower()),
            MembershipType.STANDARD,
        )
        member = self.library.register_member(name, email, phone, membership)
        print(f"\nSuccess: Registered {member}. Membership ID: {member.id}")

    def _issue_book(self) -> None:
        print("\n-- Issue Book --")
        isbn = input("Book ISBN: ").strip()
        member_id = input("Member ID: ").strip()
        loan = self.library.issue_book(isbn, member_id)
        print(f"\nSuccess: Book issued. {loan}")

    def _return_book(self) -> None:
        print("\n-- Return Book --")
        loan_id = input("Loan ID: ").strip()
        fine = self.library.return_book(loan_id)
        if fine > 0:
            print(f"\nBook returned late. Fine incurred: ${fine:.2f}")
        else:
            print("\nSuccess: Book returned on time. No fine incurred.")

    def _view_member(self) -> None:
        member_id = input("\nMember ID: ").strip()
        member = self.library.get_member(member_id)
        print(f"\n{member}")
        print(f"Membership: {member.membership_type.label}")
        print(f"Books currently borrowed: {member.borrowed_count}")
        print(f"Outstanding fine: ${member.outstanding_fine:.2f}")
        if member.active_loans:
            print("Active loans:")
            for loan in member.active_loans:
                print(f"  - {loan}")

    def _pay_fine(self) -> None:
        member_id = input("\nMember ID: ").strip()
        member = self.library.get_member(member_id)
        print(f"Current outstanding fine: ${member.outstanding_fine:.2f}")
        if member.outstanding_fine <= 0:
            print("No outstanding fine to pay.")
            return
        amount = float(input("Payment amount: ").strip())
        remaining = member.pay_fine(amount)
        print(f"Payment accepted. Remaining balance: ${remaining:.2f}")

    def _overdue_report(self) -> None:
        overdue = self.library.overdue_loans()
        if not overdue:
            print("\nNo overdue loans. Great job!")
            return
        print(f"\n-- Overdue Loans ({len(overdue)}) --")
        for loan in overdue:
            member = self.library.get_member(loan.member_id)
            print(f"  - {loan} | Member: {member.name} | Days overdue: {loan.days_overdue}")

    def _statistics(self) -> None:
        stats = self.library.statistics()
        print("\n-- Library Statistics --")
        for key, value in stats.items():
            print(f"  {key.replace('_', ' ').title()}: {value}")


def main() -> None:
    """Entry point for running the system as a script or in a notebook cell."""
    cli = LibraryCLI()
    cli.run()


if __name__ == "__main__":
    main()


Welcome to City Central Library Management System

==================== MAIN MENU ====================
 1. Add / Restock Book
 2. Search Books
 3. View Full Catalogue
 4. Register Member
 5. Issue Book
 6. Return Book
 7. View Member Details
 8. Pay Outstanding Fine
 9. Overdue Loans Report
10. Library Statistics
 0. Exit
Enter your choice: 1

-- Add / Restock Book --
Title: Harry Potter
Author: J. K. Rowling
ISBN: 12345
Genres: Fiction, Non-Fiction, Science, Technology, History, Biography, Other
Genre: other
Number of copies: 10
Publication year (optional): 2000

Success: 'Harry Potter' by J. K. Rowling [Other] ISBN:12345 (10/10 available)

==================== MAIN MENU ====================
 1. Add / Restock Book
 2. Search Books
 3. View Full Catalogue
 4. Register Member
 5. Issue Book
 6. Return Book
 7. View Member Details
 8. Pay Outstanding Fine
 9. Overdue Loans Report
10. Library Statistics
 0. Exit
Enter your choice: 1

-- Add / Restock Book --
Title: The Great Gatsby
Autho